## 2008 ITPA Threshold Power Database Analysis

In [21]:

import os
import numpy as np
import imas
import scipy as sp

ROOT = os.path.dirname(os.getcwd())
DIR_2008 = os.path.join(ROOT, "resources", "results", "2008")

pulse_dirs = sorted(
    os.path.join(DIR_2008, d)
    for d in os.listdir(DIR_2008)
    if d.startswith("pulse_")
)
N = len(pulse_dirs)
print(f"{N} pulses in {DIR_2008}")

1024 pulses in c:\Users\curranf\IDStools\resources\results\2008


## Load scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| PLTH | `summary/global_quantities/power_loss/value` | W |
| SPLASMA | `equilibrium/time_slice/global_quantities/surface` | m^2 |
| BT | `summary/global_quantities/b0/value` | T |
| NEL | `summary/line_average/n_e/value` | m^-3 |
| PGASA (M_eff) | `summary/volume_average/meff_hydrogenic/value` | AMU |
| IP | `summary/global_quantities/ip/value` | A |
| RGEO | `summary/global_quantities/r0/value` | m |
| AMIN | `summary/boundary/minor_radius/value` | m | 
| TOK | `summary/machine` | — |
| SELEC2007 | `summary/tag/name` | — |

In [22]:

def first_scalar(arr):
    """Return first element of array"""
    a = np.asarray(arr, dtype=float)
    if a.size == 0:
        return np.nan
    return float(a.flat[0])


PLTH    = np.full(N, np.nan)          # loss power [W]
PL    = np.full(N, np.nan)          # loss power [W]
SPLASMA = np.full(N, np.nan)          # LCFS surface area [m^2]
BT      = np.full(N, np.nan)          # vacuum B_t at R0 [T]
NEL     = np.full(N, np.nan)          # line-averaged n_e [m^-3]
MEFF    = np.full(N, np.nan)          # effective hydrogenic mass [AMU]
IP      = np.full(N, np.nan)          # plasma current [A]
RGEO    = np.full(N, np.nan)          # geometric major radius [m]
AMIN    = np.full(N, np.nan)          # minor radius [m]
TOK     = np.empty(N, dtype=object)   # tokamak name
SEL     = np.zeros(N, dtype=bool)     # SELEC2007 flag, for inclusion in Martin 2008 analysis

for i, pulse_dir in enumerate(pulse_dirs):
    uri = f"imas:hdf5?path={pulse_dir};pulse=0"
    if i % 100 == 0:
        print(f"Processing pulse {i}: {uri}")
    with imas.DBEntry(uri, "r") as entry:
        s = entry.get("summary")
        PLTH[i]    = first_scalar(s.global_quantities.power_loss.value)
        BT[i]      = np.abs(first_scalar(s.global_quantities.b0.value))
        NEL[i]     = first_scalar(s.line_average.n_e.value)
        MEFF[i]    = first_scalar(s.volume_average.meff_hydrogenic.value)
        IP[i]      = first_scalar(s.global_quantities.ip.value)
        RGEO[i]    = first_scalar(s.global_quantities.r0.value)
        AMIN[i]    = first_scalar(s.boundary.minor_radius.value)
        TOK[i]     = str(s.machine).strip()
        SEL[i]     = str(s.tag.name).strip() == "SELEC2007=True" # Array of bools.

        eq = entry.get("equilibrium")
        SPLASMA[i] = first_scalar(eq.time_slice[0].global_quantities.surface)

        t = entry.get("temporary")
        PL[i] = first_scalar(t.constant_float0d[0].value)
      
print("Done.")

print(f"  SELEC2007=True : {np.sum(SEL)}")

Processing pulse 0: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_0643;pulse=0
Processing pulse 100: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_1141;pulse=0
Processing pulse 200: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_1674;pulse=0
Processing pulse 300: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_2805;pulse=0
Processing pulse 400: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_3837;pulse=0
Processing pulse 500: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_5279;pulse=0
Processing pulse 600: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_5571;pulse=0
Processing pulse 700: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_6001;pulse=0
Processing pulse 800: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_6202;pulse=0
Processing pulse 900: imas:hdf5?path=c:\Users\curranf\IDS

## Selection Criteria

In [ ]:
mask = SEL

print(f"Total pulses        : {N}")
print(f"SELEC2007=True      : {np.sum(SEL)}")
print(f"Complete + selected : {np.sum(mask)}")
print()
print("SELEC2007=True")
for tok in np.unique(TOK[mask]):
    print(f"  {tok}: {np.sum((TOK == tok) & mask)}")

Total pulses        : 1024
SELEC2024=True      : 1024
Complete + selected : 1024

SELEC2024=True
  AUG: 175
  CMOD: 115
  D3D: 56
  JET: 562
  JFT2M: 58
  JT60U: 58


## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma$, $\chi^2 = \Sigma(\frac{P_{LH} - P_{\text{scal}}}{0.15P_{LH}})^2$

In [45]:
from scipy.optimize import curve_fit

m = SEL

Ploss_MW = np.where(np.isnan(PLTH), PL, PLTH) / 1e6  # [MW]
ne_20    = NEL  / 1e20 # [10^20 m^-3]
ip_ma    = np.abs(IP)   / 1e6 # [MA]

P    = Ploss_MW[m]
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
Amin = AMIN[m]
R = RGEO[m]


def model(X, C, alpha, beta, gamma):
    ne, Bt, S = X
    return C * ne**alpha * Bt**beta * S**gamma

X = (ne, Bt, S)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

C, alpha, beta, gamma = popt
perr = np.sqrt(np.diag(pcov))

print(f"C = {C:.4f} pm {perr[0]:.4f}   (paper  0.0500 pm  0.0014)")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}   (paper 0.689 pm 0.018)")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}   (paper  0.851  pm 0.016)")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}   (paper  0.885  pm 0.010)")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit:.3f}")


C = 0.0500 pm 0.0014   (paper  0.0500 pm  0.0014)
alpha  = 0.689 pm 0.018   (paper 0.689 pm 0.018)
beta = 0.851 pm 0.016   (paper  0.851  pm 0.016)
gamma = 0.885 pm 0.010   (paper  0.885  pm 0.010)
RMSE_fitnorm: 0.442



## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma I_p^\delta$, $\chi^2 = \Sigma(\frac{P_{LH} - P_{\text{scal}}}{0.15P_{LH}})^2$

In [38]:
def model(X, C, alpha, beta, gamma, delta):
    ne, Bt, S, ip_ma = X
    return C * ne**alpha * Bt**beta * S**gamma * ip_ma ** delta

X = (ne, Bt, S, ip_ma)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

C, alpha, beta, gamma, delta = popt
perr = np.sqrt(np.diag(pcov))

print(f"C = {C:.4f} pm {perr[0]:.4f}")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}")
print(f"delta = {delta:.3f} pm {perr[4]:.3f}")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit:.3f} (paper  0.207)")

C = 0.0272 pm 0.0028
alpha  = 0.746 pm 0.020
beta = 1.000 pm 0.029
gamma = 1.025 pm 0.025
delta = -0.169 pm 0.028
RMSE_fitnorm: 0.444 (paper  0.207)


## $P_{LH} = Cn_e^\alpha S^\gamma B_{\text{pol}}^\epsilon$, $\chi^2 = \Sigma(\frac{P_{LH} - P_{\text{scal}}}{0.15P_{LH}})^2$

In [42]:
def model(X, C, alpha, gamma, epsilon):
    ne, S, Bpol = X
    return C * ne**alpha * S**gamma * Bpol ** epsilon

ip = ip_ma * 1e6
mu_0 = sp.constants.mu_0
Bpol = 2*np.pi * mu_0 * ip * R / S

X = (ne, S, Bpol)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

C, alpha, gamma, epsilon = popt
perr = np.sqrt(np.diag(pcov))

print(f"C = {C:.4f} pm {perr[0]:.4f}")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}")
print(f"gamma = {gamma:.3f} pm {perr[2]:.3f}")
print(f"epsilon = {epsilon:.3f} pm {perr[3]:.3f}")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit:.3f} (paper  0.207)")

C = 0.4048 pm 0.0174
alpha  = 0.649 pm 0.020
gamma = 0.761 pm 0.012
epsilon = 0.769 pm 0.016
RMSE_fitnorm: 0.467 (paper  0.207)


In [36]:
not2 = 0

for i, meff in enumerate(Meff):
    if meff != 2: 
        not2+=1
print(f"Proportion of non-2 Meff: {not2} / {len(Meff)}")

na = 0

for i, meff in enumerate(ip_ma):
    if not(type(ip_ma[i]) == np.float64): 
        na+=1
print(f"Proportion of missing ip: {na} / {len(ip_ma)}")

Proportion of non-2 Meff: 115 / 1024
Proportion of missing ip: 0 / 1024


## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma$ (Log-space OLS)

In [ ]:
y = np.log(P)
A = np.column_stack([np.ones_like(ne), np.log(ne), np.log(Bt), np.log(S)])
coef, *_ = np.linalg.lstsq(A, y, rcond=None)

C     = np.exp(coef[0])
alpha = coef[1]   # ne exponent   (paper 0.717)
beta  = coef[2]   # Bt exponent   (paper 0.803)
gamma = coef[3]   # S  exponent   (paper 0.941)

resid    = y - A @ coef
rms_log  = np.sqrt(np.mean(resid**2)) 
dof     = len(y) - A.shape[1]            # n - k degrees of freedom
sigma2  = (resid @ resid) / dof          # residual variance
cov     = sigma2 * np.linalg.inv(A.T @ A)
perr    = np.sqrt(np.diag(cov))          # standard errors on coef

print(f"C = {C:.4f} pm {perr[0]:.4f}   (paper  0.0488 pm  0.057)")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}   (paper 0.717 pm 0.035)")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}   (paper  0.803  pm 0.032)")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}   (paper  0.941  pm 0.019)")
print(rms_log)



C = 0.0488 pm 0.0575   (paper  0.0488 pm  0.057)
alpha  = 0.717 pm 0.035   (paper 0.717 pm 0.035)
beta = 0.803 pm 0.032   (paper  0.803  pm 0.032)
gamma = 0.941 pm 0.019   (paper  0.941  pm 0.019)
0.30708712918570275



## $P_{LH} = Cn_e^\alpha B_t^\beta R^\gamma a^\delta$ (Log-space OLS)

In [ ]:
A = np.column_stack([np.ones_like(ne), np.log(ne), np.log(Bt), np.log(Amin), np.log(R)])
coef, *_ = np.linalg.lstsq(A, y, rcond=None)

C     = np.exp(coef[0])
alpha = coef[1]   # ne exponent
beta  = coef[2]   # Bt exponent
gamma = coef[3]   # R  exponent
delta = coef[4]   # a exponent

resid    = y - A @ coef
rms_log  = np.sqrt(np.mean(resid**2)) 
dof     = len(y) - A.shape[1]            # n - k degrees of freedom
sigma2  = (resid @ resid) / dof          # residual variance
cov     = sigma2 * np.linalg.inv(A.T @ A)
perr    = np.sqrt(np.diag(cov))          # standard errors on coef

print(f"C = {C:.4f} pm {perr[0]:.4f}   (paper  2.15 pm  0.107)")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}   (paper 0.782 pm 0.037)")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}   (paper  0.772  pm 0.031)")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}   (paper  0.975  pm 0.08)")
print(f"delta = {delta:.3f} pm {perr[4]:.3f}   (paper  0.999  pm 0.101)")


print(rms_log)

C = 2.1483 pm 0.1071   (paper  2.15 pm  0.107)
alpha  = 0.782 pm 0.037   (paper 0.782 pm 0.037)
beta = 0.772 pm 0.031   (paper  0.772  pm 0.031)
gamma = 0.975 pm 0.080   (paper  0.975  pm 0.08)
delta = 0.999 pm 0.101   (paper  0.999  pm 0.101)
0.29408594930977877


## Correlation matrix

In [48]:
labels = ["ne", "Bt", "ip", "Bpol"]
data = np.vstack([ne, Bt, ip, Bpol])
corr = np.corrcoef(data)

print(corr)
print()
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        print(f"corr({labels[i]}, {labels[j]}) = {corr[i, j]:.4f}")

[[ 1.          0.80350431 -0.34944893  0.69518355]
 [ 0.80350431  1.          0.07192087  0.91305434]
 [-0.34944893  0.07192087  1.          0.33884267]
 [ 0.69518355  0.91305434  0.33884267  1.        ]]

corr(ne, Bt) = 0.8035
corr(ne, ip) = -0.3494
corr(ne, Bpol) = 0.6952
corr(Bt, ip) = 0.0719
corr(Bt, Bpol) = 0.9131
corr(ip, Bpol) = 0.3388
